In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index

# =====================================================================
# RPY2 & R INTERFACE SETUP 
# =====================================================================
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

# Import R survival libraries
survival_r = importr("survival")
survminer_r = importr("survminer")

warnings.filterwarnings('ignore')

# =====================================================================
# 1. SETUP
# =====================================================================
sns.set_theme(style="whitegrid", rc={"grid.color": "#f0f0f0", "grid.linestyle": "-"})
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['pdf.fonttype'] = 42

surv_colors = {
    'Low Risk Profile': '#0072B2',
    'High Risk Profile': '#D55E00'
}
c_neutral = "#222222"

MODEL_FORMULA = "coef_z + pHSC_z"

# =====================================================================
# 2. HELPER FUNCTION
# =====================================================================
def get_col_name(df, possible_names):
    """Returns the first column name that exists in the dataframe."""
    for name in possible_names:
        if name in df.columns:
            return name
    raise KeyError(f"None of {possible_names} found in columns: {df.columns.tolist()}")

# =====================================================================
# 3. DATA LOADING & PREPARATION 
# =====================================================================
# 3A. Moffitt Loading
moffitt_params = pd.read_csv("IntermediateResults/moffitt_weighted_params.csv")
moffitt_labs = pd.read_excel("InputData/moffitt_cohort_OS_labs.xlsx")

sct_col_moffitt = get_col_name(moffitt_labs, ['Allo-SCT', 'AlloSCT', 'allo_sct'])
moffitt_labs = moffitt_labs.loc[moffitt_labs[sct_col_moffitt] == 0].copy()

id_col_moffitt = get_col_name(moffitt_labs, ['Subject_ID', 'Patient_ID', 'Study_subject', 'ID'])

# Standardize IDs before merge to avoid dropped rows
moffitt_labs[id_col_moffitt] = moffitt_labs[id_col_moffitt].astype(str).str.strip()
moffitt_params['ID'] = moffitt_params['ID'].astype(str).str.strip()
merged_moffitt = pd.merge(moffitt_labs, moffitt_params, left_on=id_col_moffitt, right_on="ID")

# 3B. Venex Loading
venex_params = pd.read_csv("IntermediateResults/venex_weighted_params.csv")
venex_labs = pd.read_excel("InputData/venex_cohort_OS_labs_with_mutations.xlsx")

id_col_venex = get_col_name(venex_labs, ['Study_subject', 'Patient_ID', 'Subject_ID', 'ID'])

# Standardize IDs before merge to avoid dropped rows
venex_labs[id_col_venex] = venex_labs[id_col_venex].astype(str).str.strip()
venex_params['ID'] = venex_params['ID'].astype(str).str.strip()
merged_venex = pd.merge(venex_labs, venex_params, left_on=id_col_venex, right_on="ID")

# 3C. Clean Standardized Extraction
os_time_m = get_col_name(merged_moffitt, ['OS (days)', 'OS_time', 'OS'])
os_stat_m = get_col_name(merged_moffitt, ['Vital_status', 'OS_status', 'Status'])

df_moffitt = pd.DataFrame({
    'Patient_ID': merged_moffitt[id_col_moffitt],
    'OS_time': pd.to_numeric(merged_moffitt[os_time_m], errors='coerce'),
    'OS_status': pd.to_numeric(merged_moffitt[os_stat_m], errors='coerce').fillna(0).astype(int),
    'pHSC_raw': pd.to_numeric(merged_moffitt['pHSC_weighted'], errors='coerce'),
    'coef_raw': pd.to_numeric(merged_moffitt['coef_weighted'], errors='coerce')
}).dropna().reset_index(drop=True)

os_time_v = get_col_name(merged_venex, ['OS_time', 'OS (days)', 'OS'])
os_stat_v = get_col_name(merged_venex, ['OS_status', 'Vital_status', 'Status'])

df_venex = pd.DataFrame({
    'Patient_ID': merged_venex[id_col_venex],
    'OS_time': pd.to_numeric(merged_venex[os_time_v], errors='coerce'),
    'OS_status': pd.to_numeric(merged_venex[os_stat_v], errors='coerce').fillna(0).astype(int),
    'pHSC_raw': pd.to_numeric(merged_venex['pHSC_weighted'], errors='coerce'),
    'coef_raw': pd.to_numeric(merged_venex['coef_weighted'], errors='coerce')
}).dropna().reset_index(drop=True)

print(f"Loaded Moffitt Cohort: {len(df_moffitt)} patients ({int(df_moffitt['OS_status'].sum())} events)")
print(f"Loaded VENEX Cohort:   {len(df_venex)} patients ({int(df_venex['OS_status'].sum())} events)")
# =====================================================================
# 3D. Administrative Right-Censoring (Match Moffitt follow-up to VenEx)
# =====================================================================
# 1. Determine the maximum follow-up time in the VenEx trial
max_venex_time = df_venex['OS_time'].max()
print(f"Maximum follow-up time in VenEx cohort: {max_venex_time} days")

# 2. Apply  right-censoring to the Moffitt cohort
# Find Moffitt patients whose survival time exceeds the VenEx maximum
moffitt_exceeds_mask = df_moffitt['OS_time'] > max_venex_time

# Cap their survival time to the VenEx maximum
df_moffitt.loc[moffitt_exceeds_mask, 'OS_time'] = max_venex_time

# Change their status to 0 (censored) because they were alive at this new cutoff
df_moffitt.loc[moffitt_exceeds_mask, 'OS_status'] = 0

print(f"Right-censored {moffitt_exceeds_mask.sum()} Moffitt patients to {max_venex_time} days.")
# =====================================================================
# 4. R SURVMINER CUTPOINT WRAPPER
# =====================================================================
def find_optimal_cutpoint(df, score_col, time_col, event_col, minprop=0.1):
    """
    Directly calls R's survminer::surv_cutpoint() via rpy2 using localconverter.
    Returns the discovered cutpoint threshold.
    """
    sub_df = df[[time_col, event_col, score_col]].dropna().copy()

    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(sub_df)

    ro.globalenv['r_df'] = r_df

    r_code = f"""
    res_cut <- survminer::surv_cutpoint(
        data = r_df,
        time = "{time_col}",
        event = "{event_col}",
        variables = "{score_col}",
        minprop = {minprop}
    )
    res_cut$cutpoint$cutpoint
    """
    r_res = ro.r(r_code)
    return float(r_res[0])

# =====================================================================
# 5. EXPERIMENTS SETUP
# =====================================================================
experiments = [
    {
        "panel_letter": "A",
        "subtitle": "Train Moffitt → Test Moffitt",
        "train_df": df_moffitt.copy(),
        "test_df": df_moffitt.copy(),
    },
    {
        "panel_letter": "B",
        "subtitle": "Train VenEx → Test Moffitt",
        "train_df": df_venex.copy(),
        "test_df": df_moffitt.copy(),
    },
    {
        "panel_letter": "C",
        "subtitle": "Train Moffitt → Test VenEx",
        "train_df": df_moffitt.copy(),
        "test_df": df_venex.copy(),
    }
]

# =====================================================================
# 6. FIGURE RENDERING 
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(24, 7.5), sharey=True)

for ax, exp in zip(axes, experiments):
    train_data = exp["train_df"].copy()
    test_data = exp["test_df"].copy()
    
    # Train-based normalization to prevent data leakage
    for param in ['pHSC', 'coef']:
        raw_col = f"{param}_raw"
        z_col = f"{param}_z"
        
        train_mean = train_data[raw_col].mean()
        train_std = train_data[raw_col].std() if train_data[raw_col].std() > 0 else 1.0
        
        train_data[z_col] = (train_data[raw_col] - train_mean) / train_std
        test_data[z_col] = (test_data[raw_col] - train_mean) / train_std
    
    # Fit Cox model on training data
    cph = CoxPHFitter(penalizer=0.01) 
    cph.fit(train_data, duration_col='OS_time', event_col='OS_status', formula=MODEL_FORMULA)
    
    # Predict hazard scores for BOTH sets
    train_data['score_mech'] = cph.predict_log_partial_hazard(train_data)
    test_data['score_mech'] = cph.predict_log_partial_hazard(test_data)
    
    c_idx = concordance_index(test_data['OS_time'], -test_data['score_mech'], test_data['OS_status'])
    
    # Find optimal cutpoint on training data via R survminer
    optimal_train_cut = find_optimal_cutpoint(train_data, 'score_mech', 'OS_time', 'OS_status', minprop=0.1)
    
    # Absolute cutpoint mapping on test cohort
    test_data['final_group'] = np.where(test_data['score_mech'] <= optimal_train_cut, 'Low Risk Profile', 'High Risk Profile')
    
    try:
        res = logrank_test(
            test_data.loc[test_data['final_group'] == 'Low Risk Profile', 'OS_time'],
            test_data.loc[test_data['final_group'] == 'High Risk Profile', 'OS_time'],
            test_data.loc[test_data['final_group'] == 'Low Risk Profile', 'OS_status'],
            test_data.loc[test_data['final_group'] == 'High Risk Profile', 'OS_status']
        )
        p_val = res.p_value
    except Exception:
        p_val = 1.0

    # Plot Kaplan-Meier curves
    for name in ['Low Risk Profile', 'High Risk Profile']:
        mask = test_data['final_group'] == name
        if mask.sum() == 0: 
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(
            durations=test_data.loc[mask, 'OS_time'], 
            event_observed=test_data.loc[mask, 'OS_status'], 
            label=f"{name} (N = {mask.sum()})"
        )
        kmf.plot_survival_function(
            ax=ax, color=surv_colors[name],
            linestyle="-" if name == 'Low Risk Profile' else "--",
            linewidth=3.4, ci_show=False, show_censors=True,
            censor_styles={'ms': 8, 'marker': '+' if name == 'Low Risk Profile' else 'x'}
        )
        
    ax.set_xlabel('Time Since Treatment Initiation (Days)', fontsize=17, fontweight='bold', labelpad=10)
    ax.set_xlim(0, test_data['OS_time'].max() * 1.03)
    ax.set_ylim(-0.02, 1.02)
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.text(-0.02, 1.05, exp["panel_letter"], transform=ax.transAxes, fontsize=26, fontweight='bold', va='top', ha='right')
    
    p_string = f"Log-rank $p = {p_val:.4e}$" if p_val < 0.001 else f"Log-rank $p = {p_val:.4f}$"
    badge_text = f"{exp['subtitle']}\n\nC-Index = {c_idx:.3f}\n{p_string}"
    ax.text(0.45, 0.58, badge_text, transform=ax.transAxes, fontsize=14, fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.92, edgecolor='#cccccc', boxstyle='round,pad=0.6'))
    ax.legend(loc='lower left', frameon=True, fontsize=14, facecolor='white', edgecolor='#cccccc', borderpad=0.6)

axes[0].set_ylabel('Overall Survival Probability', fontsize=18, fontweight='bold', labelpad=12)

plt.tight_layout()
os.makedirs('Figures', exist_ok=True)
plt.savefig('Figures/Figure3.png', dpi=300, bbox_inches='tight')
plt.savefig('Figures/Figure3.pdf', dpi=300, bbox_inches='tight')
plt.show()